In [ ]:
!pip install -U causal-conv1d
!pip install bitsandbytes
!pip install datasets evaluate accelerate
!pip install --no-build-isolation --no-cache-dir -U mamba-ssm

In [ ]:
import torch
from torch import nn
from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss
from transformers.models.mamba.modeling_mamba import (
    MambaPreTrainedModel, 
    MambaModel,
    MambaCache,
)
from transformers.modeling_outputs import SequenceClassifierOutputWithPast
from typing import List, Optional, Tuple, Union
from transformers.utils import (
    ModelOutput,
    add_start_docstrings,
    add_start_docstrings_to_model_forward,
    add_code_sample_docstrings,
)
from dataclasses import dataclass


_CHECKPOINT_FOR_DOC = "state-spaces/mamba-130m-hf"
_CONFIG_FOR_DOC = "MambaConfig"


@dataclass
class MambaSequenceClassifierOutput(ModelOutput):
    """
    Base class for outputs of sentence classification models.

    Args:
        loss (`torch.FloatTensor` of shape `(1,)`, *optional*, returned when `labels` is provided):
            Classification (or regression if config.num_labels==1) loss.
        logits (`torch.FloatTensor` of shape `(batch_size, config.num_labels)`):
            Classification (or regression if config.num_labels==1) scores (before SoftMax).
        cache_params (list of five `torch.FloatTensor` of shape `(batch_size, hidden_size, num_hidden_layers)`):
            The state of the model at the last time step. Can be used in a forward method with the next `input_ids` to
            avoid providing the old `input_ids`.
        hidden_states (`tuple(torch.FloatTensor)`, *optional*, returned when `output_hidden_states=True` is passed or when `config.output_hidden_states=True`):
            Tuple of `torch.FloatTensor` (one for the output of the embeddings, if the model has an embedding layer, +
            one for the output of each layer) of shape `(batch_size, sequence_length, hidden_size)`.

            Hidden-states of the model at the output of each layer plus the optional initial embedding outputs.
    """

    loss: Optional[torch.FloatTensor] = None
    logits: torch.FloatTensor = None
    # cache_params: Optional[MambaCache] = None,
    cache_params: Optional[List[torch.FloatTensor]] = None
    # cache_params: Optional[Tuple[Tuple[torch.FloatTensor]]] = None
    hidden_states: Optional[Tuple[torch.FloatTensor, ...]] = None
    
    
class MambaClassificationHead(nn.Module):
    """Head for sentence-level classification tasks."""

    def __init__(self, config):
        super().__init__()
        # self.activation = ACT2FN[config.hidden_act]
        # self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        # self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels, bias=False)

        # module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
        self.out_proj.weight.data.normal_(mean=0.0, std=config.initializer_range)

        self.config = config

    def forward(self, features, **kwargs):
        # x = features[:, 0, :]  # take <s> token (equiv. to [CLS])
        # x = self.dropout(x)
        # x = self.dense(x)
        # x = self.activation(x)
        # x = self.dropout(x)
        x = features
        x = self.out_proj(x)
        return x

class MambaForSequenceClassification(MambaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        # self.embeddings = nn.Embedding(config.vocab_size, config.hidden_size)
        self.backbone = MambaModel(config)
        # self.classifier = MambaClassificationHead(config)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels, bias=False)
        # self.score = nn.Linear(config.hidden_size, config.num_labels, bias=False)
        
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Initialize weights and apply final processing
        self.post_init()

    @add_code_sample_docstrings(
        checkpoint=_CHECKPOINT_FOR_DOC,
        output_type=MambaSequenceClassifierOutput,
        config_class=_CONFIG_FOR_DOC,
    )
    def forward(
        self,
        input_ids: Optional[torch.LongTensor] = None,
        inputs_embeds: Optional[torch.FloatTensor] = None,
        cache_params: Optional[MambaCache] = None,
        use_cache: Optional[bool] = None,
        labels: Optional[torch.LongTensor] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
        **kwargs,
    ) -> Union[Tuple, MambaSequenceClassifierOutput]:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss.
            Indices should be in `[0, ..., config.num_labels - 1]`.
            If `config.num_labels == 1` a regression loss is computed (Mean-Square loss),
            If `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
        """
        # use_cache = use_cache if use_cache is not None else (self.config.use_cache if not self.training else False)
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        # if inputs_embeds is None:
        #     inputs_embeds = self.backbone.embeddings(input_ids)

        # if self.backbone.gradient_checkpointing and self.training and use_cache:
        #     use_cache = False

        # if cache_params is None and use_cache:
        #     cache_params = MambaCache(
        #         self.config, inputs_embeds.size(0), device=inputs_embeds.device, dtype=inputs_embeds.dtype
        #     )

        mamba_outputs = self.backbone(
            input_ids,
            cache_params=cache_params,
            use_cache=use_cache,
            inputs_embeds=inputs_embeds,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        hidden_states = mamba_outputs[0]
        logits = self.classifier(hidden_states)

        if input_ids is not None:
            batch_size, sequence_length = input_ids.shape[:2]
        else:
            batch_size, sequence_length = inputs_embeds.shape[:2]
        assert (
            self.config.pad_token_id is not None or batch_size == 1
        ), "Cannot handle batch sizes > 1 if no padding token is defined."
        
        if self.config.pad_token_id is None:
            sequence_lengths = -1
        else:
            if input_ids is not None:
                # if no pad token found, use modulo instead of reverse indexing for ONNX compatibility
                sequence_lengths = torch.eq(input_ids, self.config.pad_token_id).int().argmax(-1) - 1
                sequence_lengths = sequence_lengths % input_ids.shape[-1]
                sequence_lengths = sequence_lengths.to(logits.device)
            else:
                sequence_lengths = -1
                print(
                    f"{self.__class__.__name__} will not detect padding tokens in `inputs_embeds`. Results may be "
                    "unexpected if using padding tokens in conjunction with `inputs_embeds.`"
                )

        pooled_logits = logits[torch.arange(batch_size, device=logits.device), sequence_lengths]

        loss = None
        if labels is not None:
            if self.config.problem_type is None:
                if self.num_labels == 1:
                    self.config.problem_type = "regression"
                elif self.num_labels > 1 and (labels.dtype == torch.long or labels.dtype == torch.int):
                    self.config.problem_type = "single_label_classification"
                else:
                    self.config.problem_type = "multi_label_classification"

            if self.config.problem_type == "regression":
                loss_fct = MSELoss()
                if self.num_labels == 1:
                    loss = loss_fct(pooled_logits.squeeze(), labels.squeeze())
                else:
                    loss = loss_fct(pooled_logits, labels)
            elif self.config.problem_type == "single_label_classification":
                loss_fct = CrossEntropyLoss()
                loss = loss_fct(pooled_logits.view(-1, self.num_labels), labels.view(-1))
            elif self.config.problem_type == "multi_label_classification":
                loss_fct = BCEWithLogitsLoss()
                loss = loss_fct(pooled_logits, labels)
        
        # if use_cache:
        #     cache_params.seqlen_offset += inputs_embeds.shape[1]
                
        if not return_dict:
            output = (pooled_logits,) + mamba_outputs[1:]
            return ((loss,) + output) if loss is not None else output

        return MambaSequenceClassifierOutput(
            loss=loss,
            logits=pooled_logits,
            cache_params=mamba_outputs.cache_params,
            hidden_states=mamba_outputs.hidden_states,
        )
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import load_dataset
from mamba_ssm import selective_scan_fn
from google.colab import drive
from peft import LoraConfig, get_peft_model, TaskType
import pandas as pd
from datasets import Dataset, DatasetDict
import os
import evaluate
import glob
import inspect, os
import math
import torch
os.environ["HF_DATASETS_CACHE"] = "/content/hf_cache"
MODEL_NAME = "state-spaces/mamba-130m-hf"
NUM_LABELS = 2
TRAIN_CSV = "/content/train_clean.csv"
VAL_CSV   = "/content/val_clean.csv"
OUTPUT_DIR = "mamba_base_lora"
max_length = 128
BATCH_SIZE = 32
NUM_EPOCHS = 18
LR = 2e-4
LORA_R = 12
LORA_ALPHA = 32
LORA_DROP = 0.05

In [ ]:
def train():

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    id2label = {0: "NEGATIVE", 1: "POSITIVE"}
    label2id = {"NEGATIVE": 0, "POSITIVE": 1}

    model = MambaForSequenceClassification.from_pretrained(MODEL_NAME, num_labels = NUM_LABELS, use_cache = False, id2label = id2label, label2id = label2id)
    model.to("cuda")

    train_df = pd.read_csv(TRAIN_CSV)
    val_df   = pd.read_csv(VAL_CSV)

    train_ds = Dataset.from_pandas(train_df)
    val_ds   = Dataset.from_pandas(val_df)

    raw_dataset = DatasetDict({"train": train_ds, "validation": val_ds})
    torch.backends.cudnn.benchmark = True

    def preprocess(examples):
        texts = [str(x) for x in examples["text"]]
        enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_length,
        )
        enc["labels"] = [int(x) for x in examples["label"]]
        return enc

    dataset = raw_dataset.map(
        preprocess,
        batched=True,
        remove_columns=["text", "label"],
    )

    peft_config = LoraConfig(
        task_type = TaskType.SEQ_CLS,
        target_modules = ["in_proj", "out_proj", "x_proj", "proj_in", "proj_out"],
        r = LORA_R,
        lora_alpha = LORA_ALPHA,
        lora_dropout = LORA_DROP,
        bias = 'none'
    )

    final_model = get_peft_model(model, peft_config)
    final_model.to("cuda")
    print(" % OF TRAINING")
    final_model.print_trainable_parameters()

    metric_acc = evaluate.load("accuracy")
    metric_f1  = evaluate.load("f1")
    metric_precision = evaluate.load("precision")
    metric_recall = evaluate.load("recall")

    def compute_metrics(p):
      preds = p.predictions.argmax(-1)
      return {
          "accuracy": metric_acc.compute(predictions = preds, references = p.label_ids)["accuracy"],
          "f1":       metric_f1.compute(predictions = preds, references = p.label_ids, average = "binary")["f1"],
          "precision":       metric_precision.compute(predictions = preds, references = p.label_ids, average="binary")["precision"],
          "recall":       metric_recall.compute(predictions = preds, references = p.label_ids, average="binary")["recall"],
      }

    final_model.gradient_checkpointing_enable()
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32  = True
    drive.mount('/content/drive')
    OUTPUT_DIR_DRIVE = "/content/drive/MyDrive/mamba_checkpoints"
    import os
    os.makedirs(OUTPUT_DIR_DRIVE, exist_ok=True)
    #final_model = torch.compile(final_model, mode="default", fullgraph=False)
    training_args = TrainingArguments(
        output_dir                  = OUTPUT_DIR_DRIVE,
        per_device_train_batch_size = BATCH_SIZE,
        learning_rate               = LR,
        gradient_accumulation_steps = 8,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        dataloader_num_workers      = 2,
        warmup_ratio                = 0.1,
        lr_scheduler_type           = "cosine",
        dataloader_pin_memory       = True,
        bf16                        = True,
        optim                       = "adamw_torch_fused",
        max_grad_norm               = 1.0,
        fp16                        = False,
        num_train_epochs            = NUM_EPOCHS,
        logging_strategy            = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "f1",
        remove_unused_columns       = False,
        greater_is_better           = True,
        report_to                   = "none",
        label_names                 = ["labels"]
    )

    #final_model = torch.compile(final_model)
    with torch.no_grad():
      torch.nn.init.kaiming_uniform_(final_model.classifier.weight, a = math.sqrt(5))

    trainer = Trainer(
        model               = final_model,
        args                = training_args,
        train_dataset       = dataset["train"],
        tokenizer           = tokenizer,
        eval_dataset        = dataset["validation"],
        data_collator       = DataCollatorWithPadding(tokenizer, return_tensors = 'pt'),
        compute_metrics     = compute_metrics
    )

    all_ckpts = sorted(
    glob.glob(os.path.join(OUTPUT_DIR_DRIVE, "checkpoint-*")),
    key=lambda x: int(x.split("-")[-1])
    )
    if all_ckpts:
        print("🔄 Riprendo da:", all_ckpts[-1])
        trainer.train(resume_from_checkpoint=all_ckpts[-1])
    else:
        print("🔄 Nessun checkpoint trovato, inizio da zero")
        trainer.train()
    metrics = trainer.evaluate()
    print("Final evaluation:", metrics)

    trainer.save_model(os.path.join(OUTPUT_DIR_DRIVE, "final_model"))

In [ ]:
train()

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import logging
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, DataCollatorWithPadding
from peft import PeftModel
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    cohen_kappa_score,
    confusion_matrix,
    auc
)
from hf_mamba_classification import MambaForSequenceClassification

# CONFIGURATION
MODEL_NAME = "state-spaces/mamba-130m-hf"
ADAPTER_DIR = "mamba_base_lora/final_model"
BATCH_SIZE = 32
MAX_LENGTH = 128
RESULTS_DIR = "results"
DATA_DIR = Path("/content")  # adjust as needed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Set up logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

def compute_probs_from_hf_dataset(
    hf_dataset: Dataset,
    model: torch.nn.Module,
    tokenizer,
    device: torch.device
) -> np.ndarray:
    collator = DataCollatorWithPadding(tokenizer, return_tensors="pt")
    pin_memory = device.type == "cuda"
    loader = DataLoader(
        hf_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=2,
        pin_memory=pin_memory
    )
    all_probs = []
    model.eval()
    with torch.no_grad():
        for batch in loader:
            inputs = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]
            all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)


def save_results_and_plots(
    y_true: np.ndarray,
    probs: np.ndarray,
    set_name: str,
    threshold: float,
    out_dir: str = RESULTS_DIR
):
    # Create output directory
    os.makedirs(out_dir, exist_ok=True)

    # Compute predictions
    y_pred = (probs >= threshold).astype(int)

    # Compute confusion matrix components
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    # Collect metrics
    metrics = {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall (Sensitivity)": recall_score(y_true, y_pred),
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else np.nan,
        "Negative Predictive Value": tn / (tn + fn) if (tn + fn) > 0 else np.nan,
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Cohen Kappa": cohen_kappa_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, probs),
        "PR-AUC": average_precision_score(y_true, probs)
    }
    metrics_df = pd.DataFrame([metrics])
    metrics_path = os.path.join(out_dir, f"{set_name}_metrics.csv")
    metrics_df.to_csv(metrics_path, index=False)

    # Save per-sample probabilities and predictions
    results_df = pd.DataFrame({
        "true_label": y_true,
        "probability": probs,
        "pred_label": y_pred
    })
    results_path = os.path.join(out_dir, f"{set_name}_predictions.csv")
    results_df.to_csv(results_path, index=False)

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, probs)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}", linewidth=2)
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
    ax.set(
        xlabel="False Positive Rate",
        ylabel="True Positive Rate",
        title=f"ROC Curve ({set_name})"
    )
    ax.legend(loc="lower right")
    plt.tight_layout()
    roc_path = os.path.join(out_dir, f"{set_name}_roc_curve.png")
    fig.savefig(roc_path, dpi=300)
    plt.close(fig)

    # Precision-Recall Curve
    precision, recall, _ = precision_recall_curve(y_true, probs)
    pr_auc = auc(recall, precision)
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(recall, precision, label=f"AP = {pr_auc:.3f}", linewidth=2)
    ax.set(
        xlabel="Recall",
        ylabel="Precision",
        title=f"Precision–Recall Curve ({set_name})"
    )
    ax.legend(loc="upper right")
    plt.tight_layout()
    pr_path = os.path.join(out_dir, f"{set_name}_pr_curve.png")
    fig.savefig(pr_path, dpi=300)
    plt.close(fig)

    logger.info(f"[Saved] {set_name} metrics -> {metrics_path}")
    logger.info(f"[Saved] {set_name} predictions -> {results_path}")
    logger.info(f"[Saved] {set_name} ROC curve -> {roc_path}")
    logger.info(f"[Saved] {set_name} PR curve -> {pr_path}")


def main():
    # Initialize tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    base_model = MambaForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, use_cache=False
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR).to(device)

    # Load data
    test_df = pd.read_csv(DATA_DIR / "test_clean.csv")
    val_df = pd.read_csv(DATA_DIR / "val_clean.csv")

    test_ds = Dataset.from_pandas(test_df)
    val_ds = Dataset.from_pandas(val_df)
    raw_dataset = DatasetDict({"test": test_ds, "val": val_ds})

    # Preprocessing function
    def preprocess(examples):
        texts = [str(x) for x in examples["text"]]
        encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH
        )
        encodings["labels"] = [int(x) for x in examples["label"]]
        return encodings

    # Tokenize datasets
    dataset = raw_dataset.map(
        preprocess,
        batched=True,
        remove_columns=["text", "label"]
    )

    # Extract splits
    y_val = np.array(dataset["val"]["labels"])
    y_test = np.array(dataset["test"]["labels"])

    # Compute validation probabilities and find optimal threshold
    probs_val = compute_probs_from_hf_dataset(dataset["val"], model, tokenizer, device)
    fpr, tpr, thresholds = roc_curve(y_val, probs_val)
    youden_j = tpr - fpr
    best_idx = np.argmax(youden_j)
    best_thresh = thresholds[best_idx]
    logger.info(f"Optimal threshold from validation: {best_thresh:.3f}")

    # Compute test probabilities
    probs_test = compute_probs_from_hf_dataset(dataset["test"], model, tokenizer, device)

    # Save results and plots
    save_results_and_plots(y_test, probs_test, set_name="test_youden", threshold=best_thresh)
    save_results_and_plots(y_test, probs_test, set_name="test_standard", threshold=0.5)

In [ ]:
test()
!zip -r results.zip /content/results